In [1]:
import torch
from torchvision import datasets, transforms
from torchvision.models import vit_b_16,vit_b_32
import torch.optim as optim
import torch.nn as nn
from torch.utils.data import DataLoader,random_split
from sklearn.metrics import classification_report

NUM_CLASSES = 2
EPOCHS = 10
LEARNING_RATE = 1e-4


In [2]:
import os
from torch.cuda import manual_seed
from google.colab import drive

drive.mount("/content/drive")

path = '/content/drive/MyDrive/base_data'

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

full_datasets = datasets.ImageFolder(path, transform = transform)

train_size = int(0.8*len(full_datasets))
val_size = int(0.1*len(full_datasets))
test_size = len(full_datasets)- train_size - val_size

train_dataset,val_dataset,test_dataset = random_split(full_datasets,[train_size,val_size,test_size],generator=torch.Generator().manual_seed(42))


Mounted at /content/drive


In [3]:
#Data Loader

train_loader = DataLoader(
    train_dataset,
    batch_size= 32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

In [4]:
class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()

        self.conv = nn.Conv2d(
            2,
            1,
            kernel_size=7,
            padding=3,
            bias=False
        )

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):

        avg_out = torch.mean(x, dim=1, keepdim=True)

        max_out, _ = torch.max(
            x,
            dim=1,
            keepdim=True
        )

        x_cat = torch.cat(
            [avg_out, max_out],
            dim=1
        )

        attention = self.sigmoid(
            self.conv(x_cat)
        )

        return x * attention



In [5]:
#Window attention
class WindowAttention(nn.Module):

    def __init__(
        self,
        dim=768,
        num_heads=8
    ):
        super().__init__()

        self.attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=num_heads,
            batch_first=True
        )

    def forward(self, x):

        out, _ = self.attn(
            x,
            x,
            x
        )

        return out





In [6]:
import numpy as np
from sklearn.decomposition import PCA

feat_extractor16 = vit_b_16(weights="IMAGENET1K_V1")
feat_extractor32 = vit_b_32(weights="IMAGENET1K_V1")

feat_extractor16.heads = nn.Identity()
feat_extractor32.heads = nn.Identity()

feat_extractor16 = feat_extractor16.to(device).eval()
feat_extractor32 = feat_extractor32.to(device).eval()

all_features = []

with torch.no_grad():
    for images, _ in train_loader:
        images = images.to(device)

        f16 = feat_extractor16(images)
        f32 = feat_extractor32(images)

        fused = torch.cat([f16, f32], dim=1)
        all_features.append(fused.cpu().numpy())

all_features = np.concatenate(all_features, axis=0)

pca_components = 128
pca = PCA(n_components=pca_components)
pca.fit(all_features)

del feat_extractor16, feat_extractor32, all_features
torch.cuda.empty_cache()

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:02<00:00, 132MB/s]


Downloading: "https://download.pytorch.org/models/vit_b_32-d86f8d99.pth" to /root/.cache/torch/hub/checkpoints/vit_b_32-d86f8d99.pth


100%|██████████| 337M/337M [00:02<00:00, 144MB/s]


In [7]:
class PCAFusion(nn.Module):
    def __init__(self, pca):
        super().__init__()

        mean = torch.tensor(pca.mean_, dtype=torch.float32)
        components = torch.tensor(pca.components_, dtype=torch.float32)

        self.register_buffer("mean", mean)
        self.register_buffer("components", components)

    def forward(self, x):
        x = x - self.mean
        x = x @ self.components.T
        return x

In [15]:
#Hybrid model
class HybridViT(nn.Module):

    def __init__(self,pca, num_classes=2,feat_dim=128):

        super().__init__()

        self.vit16 = vit_b_16(weights="IMAGENET1K_V1")
        self.vit32 = vit_b_32(weights="IMAGENET1K_V1")

        feat16 = self.vit16.heads.head.in_features
        feat32 = self.vit32.heads.head.in_features

        self.vit16.heads = nn.Identity()
        self.vit32.heads = nn.Identity()

        self.fusion = PCAFusion(pca)

        self.spatial_attention = SpatialAttention()

        self.window_attention = WindowAttention(
            dim=feat_dim,
            num_heads=8
        )

        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):

        feat16 = self.vit16(x)
        feat32 = self.vit32(x)

        fused = torch.cat(
            [feat16, feat32],
            dim=1
        )

        fused = self.fusion(fused)

        B = fused.size(0)

        spatial_map = fused.view(
            B,
            128,
            1,
            1
        )

        spatial_map = self.spatial_attention(
            spatial_map
        )

        fused = spatial_map.view(
            B,
            1,
            128
        )

        fused = self.window_attention(
            fused
        )

        fused = fused.squeeze(1)

        output = self.classifier(
            fused
        )

        return output



In [9]:
#MODEL

model = HybridViT(
    pca = pca,
    num_classes=NUM_CLASSES
).to(device)

criterion = nn.CrossEntropyLoss()

optimizer = optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)

In [10]:
def train_one_epoch():

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(
            outputs,
            labels
        )

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, preds = torch.max(
            outputs,
            1
        )

        total += labels.size(0)

        correct += (
            preds == labels
        ).sum().item()

    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100 * correct / total

    return epoch_loss, epoch_acc


In [11]:
def validate():

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(
                outputs,
                labels
            )

            running_loss += loss.item()

            _, preds = torch.max(
                outputs,
                1
            )

            total += labels.size(0)

            correct += (
                preds == labels
            ).sum().item()

    epoch_loss = running_loss / len(val_loader)
    epoch_acc = 100 * correct / total

    return epoch_loss, epoch_acc

In [16]:
best_val_acc = 0

for epoch in range(EPOCHS):

    train_loss, train_acc = train_one_epoch()

    val_loss, val_acc = validate()

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_acc:.2f}% "
        f"Val Loss: {val_loss:.4f} "
        f"Val Acc: {val_acc:.2f}%"
    )

    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save({
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "epoch": epoch,
            "val_acc": val_acc
        }, "best_hybrid_vit.pth")

print("\nTraining Complete")
print("Best Validation Accuracy:", best_val_acc)


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacity of 14.56 GiB of which 7.81 MiB is free. Including non-PyTorch memory, this process has 14.55 GiB memory in use. Of the allocated memory 14.00 GiB is allocated by PyTorch, and 435.37 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)

In [ ]:
model.load_state_dict(
    torch.load(
        "best_hybrid_vit.pth",
        map_location=device
    )["model"]
)

model.eval()

all_preds = []
all_labels = []

correct = 0
total = 0

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)
        labels = labels.to(device)

        outputs = model(images)

        _, preds = torch.max(
            outputs,
            1
        )

        total += labels.size(0)

        correct += (
            preds == labels
        ).sum().item()

        all_preds.extend(
            preds.cpu().numpy()
        )

        all_labels.extend(
            labels.cpu().numpy()
        )

test_accuracy = 100 * correct / total

print("\nTest Accuracy:", test_accuracy)

print("\nClassification Report:\n")

print(
    classification_report(
        all_labels,
        all_preds,
        target_names=full_datasets.classes
    )
)
